# Spiegel repair

Re-fetch title and article text for the rows whose columns shifted. Images, dates and urls in the file are intact and are not touched.

In [ ]:
# imports
import asyncio
import csv
import json
import random
import re
from pathlib import Path

import aiohttp
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.asyncio import tqdm

In [ ]:
# settings, matching the original collection
final_dir = Path(".")
source_file = final_dir / "spiegel_final_data_v2.csv"
repair_file = final_dir / "spiegel_repair.csv"
error_file = final_dir / "spiegel_repair_errors.csv"

# slower settings for the retry
concurrent_requests = 4
sleep_range = (0.8, 1.6)

timeout = 30

headers = {"User-Agent": ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) "
                          "Chrome/122.0.0.0 Safari/537.36")}

## Identify the rows to re-fetch

In [ ]:
# a row is broken when its title holds the page text instead of a headline
df = pd.read_csv(source_file, low_memory=False)
broken = df["title"].astype(str).str.len() > 300

print(f"{len(df)} rows, {broken.sum()} broken, {(~broken).sum()} intact")
broken_urls = df.loc[broken, "article_url"].dropna().unique().tolist()
print(f"{len(broken_urls)} urls to re-fetch")

## Re-fetch

In [ ]:
# read the headline from json-ld and the body from the paragraph tags, as in the original run
def extract_ld_json(soup):
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string)
        except (json.JSONDecodeError, TypeError):
            continue
        if isinstance(data, dict) and data.get("@type") in ["NewsArticle", "Article"]:
            return data
    return {}


def parse_page(url, html):
    soup = BeautifulSoup(html, "html.parser")
    ld = extract_ld_json(soup)
    title = ld.get("headline") or (soup.title.text.strip() if soup.title else "")
    paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    return {"article_url": url,
            "title": title,
            "text": "\n".join(p for p in paragraphs if len(p) > 50)}

In [ ]:
# create the output files on the first run and leave them alone afterwards
columns = ["article_url", "title", "text"]


def init_files():
    if not repair_file.exists():
        with open(repair_file, "w", newline="", encoding="utf-8-sig") as f:
            csv.DictWriter(f, fieldnames=columns).writeheader()
    if not error_file.exists():
        with open(error_file, "w", newline="", encoding="utf-8-sig") as f:
            csv.writer(f).writerow(["article_url", "error"])


# a row counts as fetched only when it carries a headline or a body
def fetched_urls():
    done = pd.read_csv(repair_file, keep_default_na=False, na_values=[])
    filled = ((done["title"].astype(str).str.strip() != "")
              | (done["text"].astype(str).str.strip() != ""))
    return set(done.loc[filled, "article_url"])


# an empty page is a failure, not a result
async def fetch_one(session, semaphore, url, csv_lock, error_lock):
    async with semaphore:
        await asyncio.sleep(random.uniform(*sleep_range))
        try:
            async with session.get(url, timeout=timeout) as response:
                response.raise_for_status()
                html = await response.text()
            row = parse_page(url, html)
            if not str(row["title"]).strip() and not str(row["text"]).strip():
                raise ValueError("empty page, nothing extracted")
            async with csv_lock:
                with open(repair_file, "a", newline="", encoding="utf-8-sig") as f:
                    csv.DictWriter(f, fieldnames=columns).writerow(row)
        except Exception as exc:
            async with error_lock:
                with open(error_file, "a", newline="", encoding="utf-8-sig") as f:
                    csv.writer(f).writerow([url, str(exc)])

# fetch every url not already in the repair file, appending each result as it arrives,
# so the cell can be re-run after a crash and picks up where it stopped
async def refetch(urls):
    init_files()
    done = fetched_urls()
    todo = [url for url in urls if url not in done]
    print(f"{len(todo)} to fetch, {len(urls) - len(todo)} already done")
    if not todo:
        return

    semaphore = asyncio.Semaphore(concurrent_requests)
    csv_lock, error_lock = asyncio.Lock(), asyncio.Lock()
    async with aiohttp.ClientSession(headers=headers) as session:
        await tqdm.gather(*[fetch_one(session, semaphore, url, csv_lock, error_lock)
                            for url in todo])

    done = fetched_urls()
    errors = set(pd.read_csv(error_file)["article_url"].dropna()) - done
    print(f"repair file holds {len(done)} rows, {len(errors)} urls still failing")

In [ ]:
await refetch(broken_urls)

Re-run the cell below as many times as needed. It fetches only what is still missing, so it is safe to run after an interruption and safe to run when everything is already done.

In [ ]:
#await refetch(broken_urls)

In [ ]:
repaired = pd.read_csv(final_dir / "spiegel_repair.csv", keep_default_na=False, na_values=[])
empty = ((repaired["title"].astype(str).str.strip() == "")
         & (repaired["text"].astype(str).str.strip() == ""))
print(f"{len(repaired)} rows, {int(empty.sum())} still empty")
print(repaired["text"].astype(str).str.len().describe().to_string())

In [ ]:
filled = repaired[~empty]
print(f"{len(filled)} filled rows, {filled['article_url'].nunique()} unique urls")
print(f"urls with no filled row at all: {len(set(repaired['article_url']) - set(filled['article_url']))}")

## Merge back

In [ ]:
# merge the re-fetched title and text back, keeping every other column as it is
df = pd.read_csv(final_dir / "spiegel_final_data_v2.csv", low_memory=False)
broken = df["title"].astype(str).str.len() > 300

repaired = pd.read_csv(final_dir / "spiegel_repair.csv", keep_default_na=False, na_values=[])
filled = repaired[(repaired["title"].astype(str).str.strip() != "")
                  | (repaired["text"].astype(str).str.strip() != "")]
filled = filled.drop_duplicates("article_url", keep="last").set_index("article_url")
print(f"{len(filled)} usable rows for {int(broken.sum())} broken articles")

fixed = df.copy()
mask = fixed["article_url"].isin(filled.index)
fixed.loc[mask, "title"] = fixed.loc[mask, "article_url"].map(filled["title"])
fixed.loc[mask, "text"] = fixed.loc[mask, "article_url"].map(filled["text"])

still_broken = fixed["title"].astype(str).str.len() > 300
print(f"repaired {int(mask.sum())}, still broken {int(still_broken.sum())}")
print(f"median title length now {fixed['title'].astype(str).str.len().median():.0f}")
print(f"text that is still a json list of image urls: "
      f"{int(fixed['text'].astype(str).str.strip().str.startswith('[\"http').sum())}")

## Re-apply the corpus rule of section 4.2

The headline filter was applied to the wrong field for the broken rows, so it has to run again on the repaired headlines.

In [ ]:
# keep only articles whose headline names russia or a russian actor
# [подставь полное немецкое выражение из Приложения вместо этой краткой формы]
russia_pattern = (r"(?i)\b(?:russland\w*|russisch\w*|russe\w*|kreml|moskau|putin|lawrow|peskow"
                  r"|schoigu|gerassimow|medwedew|sacharowa|mischustin|gazprom|rosneft|rosatom"
                  r"|transneft|lukoil|donezk|luhansk|lugansk|krim)\b")

in_corpus = fixed["title"].astype(str).str.contains(russia_pattern, regex=True, na=False)
print(f"{int(in_corpus.sum())} of {len(fixed)} headlines name russia, dropping {int((~in_corpus).sum())}")

fixed["category"] = fixed["article_url"].str.replace(
    "https://www.spiegel.de/", "", regex=False).str.split("/").str[0]
fixed[in_corpus & ~still_broken].to_csv(
    final_dir / "spiegel_final_data_v2.csv", index=False, encoding="utf-8-sig")
print("saved")